# Baseline de Detecci?n de Clones en Python (Decision Tree)

Este notebook implementa el baseline completo en formato tipo Colab:
- Reconstrucci?n de pares de snippets
- Preprocesamiento y tokenizaci?n
- Extracci?n de features l?xicas
- Balanceo del conjunto de entrenamiento
- Entrenamiento con **DecisionTreeClassifier**
- Evaluaci?n en dos tareas (`is_clone` y `clone_type`)


## 1) Configuraci?n e importaciones

En esta secci?n cargamos librer?as y m?dulos del proyecto. Tambi?n fijamos semilla para reproducibilidad.


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

from baseline.data_loading import (
    load_metadata_csv,
    validate_metadata_schema,
    clean_metadata_rows,
    grouped_train_val_test_split,
    split_statistics,
    balance_training_dataframe,
)
from baseline.snippet_reconstruction import reconstruct_pairs_from_metadata
from baseline.feature_extraction import (
    prepare_pair_text_fields,
    fit_tfidf_vectorizer,
    build_pair_features,
)
from baseline.evaluate_baseline import evaluate_predictions
from baseline.utils import imbalance_ratio

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RUTA_BASE = Path.cwd()
if (RUTA_BASE / 'DataBaseProject' / 'clone_pairs_dataset_metadata.csv').exists():
    RUTA_DATASET = RUTA_BASE / 'DataBaseProject'
elif (RUTA_BASE / 'clone_pairs_dataset_metadata.csv').exists():
    RUTA_DATASET = RUTA_BASE
else:
    raise FileNotFoundError('No se encontr? clone_pairs_dataset_metadata.csv en rutas esperadas')

RUTA_METADATA = RUTA_DATASET / 'clone_pairs_dataset_metadata.csv'
ESTRATEGIA_BALANCEO = 'undersample'  # opciones: none, undersample, oversample

print('RUTA_BASE:', RUTA_BASE)
print('RUTA_DATASET:', RUTA_DATASET)
print('RUTA_METADATA:', RUTA_METADATA)
print('ESTRATEGIA_BALANCEO:', ESTRATEGIA_BALANCEO)


## 2) Carga de metadata, validaci?n y limpieza

In [ ]:
DatosMetadata = load_metadata_csv(RUTA_METADATA)
print('Filas metadata:', len(DatosMetadata))

EsquemaOK, ErroresEsquema = validate_metadata_schema(DatosMetadata)
if not EsquemaOK:
    raise ValueError(f'Error de esquema: {ErroresEsquema}')

DatosMetadataLimpios, FilasInvalidas = clean_metadata_rows(DatosMetadata)
print('Filas validas:', len(DatosMetadataLimpios))
print('Filas invalidas:', len(FilasInvalidas))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
DatosMetadataLimpios['is_clone'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#4C78A8', '#F58518'])
axes[0].set_title('Distribucion inicial: is_clone')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Cantidad')

DatosMetadataLimpios[DatosMetadataLimpios['is_clone'] == 1]['clone_type'].value_counts().plot(kind='bar', ax=axes[1], color=['#72B7B2', '#E45756'])
axes[1].set_title('Distribucion inicial: clone_type (solo positivos)')
axes[1].set_xlabel('Tipo')
axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()

DatosMetadataLimpios.head(3)


## 3) Reconstrucci?n de snippets (`code_a`, `code_b`)

In [ ]:
DatosReconstruidos, FilasDescartadas, ResumenReconstruccion = reconstruct_pairs_from_metadata(
    DatosMetadataLimpios,
    dataset_root=RUTA_DATASET,
)

print('Resumen reconstruccion:')
print(ResumenReconstruccion)
print('')
print('Muestra de pares reconstruidos:')
display(DatosReconstruidos[['file_path', 'snippet_index_a', 'snippet_index_b', 'is_clone', 'clone_type']].head(5))

plt.figure(figsize=(7, 4))
plt.hist(DatosReconstruidos['snippet_count'], bins=30, color='#54A24B', edgecolor='black')
plt.title('Histograma de snippet_count en pares reconstruidos')
plt.xlabel('Cantidad de snippets por archivo')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()


## 4) Preprocesamiento y tokenizaci?n

In [ ]:
DatosPreparados = prepare_pair_text_fields(DatosReconstruidos)
print('Filas procesadas:', len(DatosPreparados))

display(DatosPreparados[['code_a_clean', 'code_b_clean', 'token_text_a', 'token_text_b']].head(2))

LongTokensA = DatosPreparados['tokens_a'].apply(len)
LongTokensB = DatosPreparados['tokens_b'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(LongTokensA, bins=40, color='#4C78A8', edgecolor='black')
axes[0].set_title('Histograma tokens_a')
axes[0].set_xlabel('Numero de tokens')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(LongTokensB, bins=40, color='#F58518', edgecolor='black')
axes[1].set_title('Histograma tokens_b')
axes[1].set_xlabel('Numero de tokens')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()


## 5) Funciones auxiliares para split, entrenamiento y reporte

In [ ]:
def asignar_split(df, idx_train, idx_val, idx_test, nombre_columna='split'):
    DatosSplit = df.copy()
    DatosSplit[nombre_columna] = 'unassigned'
    DatosSplit.loc[idx_train, nombre_columna] = 'train'
    DatosSplit.loc[idx_val, nombre_columna] = 'val'
    DatosSplit.loc[idx_test, nombre_columna] = 'test'
    return DatosSplit


def graficar_balanceo(info_balanceo, titulo):
    antes = info_balanceo['class_distribution_before']
    despues = info_balanceo['class_distribution_after']

    clases = sorted(set(list(antes.keys()) + list(despues.keys())))
    valores_antes = [antes.get(c, 0) for c in clases]
    valores_despues = [despues.get(c, 0) for c in clases]

    x = np.arange(len(clases))
    width = 0.35

    plt.figure(figsize=(8, 4))
    plt.bar(x - width/2, valores_antes, width=width, label='Antes', color='#4C78A8')
    plt.bar(x + width/2, valores_despues, width=width, label='Despues', color='#E45756')
    plt.xticks(x, clases)
    plt.title(titulo)
    plt.xlabel('Clase')
    plt.ylabel('Cantidad')
    plt.legend()
    plt.tight_layout()
    plt.show()


def entrenar_evaluar_decision_tree(
    DatosTask,
    columna_target,
    etiquetas,
    seed,
    estrategia_balanceo='undersample',
    graficar_roc=False,
):
    TrainRaw = DatosTask[DatosTask['split'] == 'train'].copy()
    Val = DatosTask[DatosTask['split'] == 'val'].copy()
    Test = DatosTask[DatosTask['split'] == 'test'].copy()

    TrainBalanceado, InfoBalanceo = balance_training_dataframe(
        train_df=TrainRaw,
        target_col=columna_target,
        strategy=estrategia_balanceo,
        seed=seed,
    )

    graficar_balanceo(InfoBalanceo, f'Balanceo train - {columna_target}')

    RatioDesbalance = imbalance_ratio(TrainBalanceado[columna_target])
    PesoClases = 'balanced' if RatioDesbalance >= 1.5 else None

    VectorTfIdf = fit_tfidf_vectorizer(TrainBalanceado)
    CaracteristicasTrain = build_pair_features(TrainBalanceado, VectorTfIdf)
    CaracteristicasVal = build_pair_features(Val, VectorTfIdf)
    CaracteristicasTest = build_pair_features(Test, VectorTfIdf)

    modeloArbol = DecisionTreeClassifier(
        criterion='gini',
        max_depth=12,
        min_samples_leaf=2,
        class_weight=PesoClases,
        random_state=seed,
    )
    modeloArbol.fit(CaracteristicasTrain, TrainBalanceado[columna_target])

    yVal = Val[columna_target]
    yTest = Test[columna_target]

    predVal = modeloArbol.predict(CaracteristicasVal)
    predTest = modeloArbol.predict(CaracteristicasTest)

    probaVal = modeloArbol.predict_proba(CaracteristicasVal)
    probaTest = modeloArbol.predict_proba(CaracteristicasTest)

    MetricasVal = evaluate_predictions(yVal, predVal, labels=etiquetas, y_proba=probaVal)
    MetricasTest = evaluate_predictions(yTest, predTest, labels=etiquetas, y_proba=probaTest)

    print('--- Informacion de balanceo ---')
    print(InfoBalanceo)
    print('Ratio desbalance train despues de balancear:', round(RatioDesbalance, 4))
    print('class_weight usado:', PesoClases)

    print('')
    print('--- Metricas VALIDACION ---')
    print('accuracy:', round(MetricasVal['accuracy'], 4))
    print('f1_macro:', round(MetricasVal['f1_macro'], 4))

    print('')
    print('--- Metricas TEST ---')
    print('accuracy:', round(MetricasTest['accuracy'], 4))
    print('precision_macro:', round(MetricasTest['precision_macro'], 4))
    print('recall_macro:', round(MetricasTest['recall_macro'], 4))
    print('f1_macro:', round(MetricasTest['f1_macro'], 4))

    print('')
    print('Reporte de clasificacion TEST:')
    print(MetricasTest['classification_report_text'])

    cm_val = np.array(MetricasVal['confusion_matrix'])
    cm_test = np.array(MetricasTest['confusion_matrix'])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ConfusionMatrixDisplay(cm_val, display_labels=[str(x) for x in etiquetas]).plot(ax=axes[0], colorbar=False)
    axes[0].set_title(f'{columna_target} - Matriz VAL')
    ConfusionMatrixDisplay(cm_test, display_labels=[str(x) for x in etiquetas]).plot(ax=axes[1], colorbar=False)
    axes[1].set_title(f'{columna_target} - Matriz TEST')
    plt.tight_layout()
    plt.show()

    if graficar_roc and len(etiquetas) == 2:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        RocCurveDisplay.from_predictions(yVal, probaVal[:, 1], pos_label=etiquetas[1], ax=axes[0])
        axes[0].set_title(f'{columna_target} - ROC VAL')
        RocCurveDisplay.from_predictions(yTest, probaTest[:, 1], pos_label=etiquetas[1], ax=axes[1])
        axes[1].set_title(f'{columna_target} - ROC TEST')
        plt.tight_layout()
        plt.show()

    return {
        'info_balanceo': InfoBalanceo,
        'metricas_val': MetricasVal,
        'metricas_test': MetricasTest,
        'modelo': modeloArbol,
    }


## 6) Task A: `is_clone` (0/1)

Se construye split por `problem_id`, se balancea train y se entrena Decision Tree.


In [ ]:
SplitTaskA = grouped_train_val_test_split(
    df=DatosPreparados,
    group_col='problem_id',
    target_col='is_clone',
    seed=SEED,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
)

DatosTaskA = asignar_split(
    DatosPreparados,
    idx_train=SplitTaskA.train_idx,
    idx_val=SplitTaskA.val_idx,
    idx_test=SplitTaskA.test_idx,
)

print('Estadisticas split Task A:')
StatsA = split_statistics(DatosTaskA, 'split', 'is_clone', 'problem_id')
for fila in StatsA:
    print(fila)

conteos_split_a = DatosTaskA.groupby(['split', 'is_clone']).size().unstack(fill_value=0)
conteos_split_a.plot(kind='bar', figsize=(8,4), color=['#4C78A8', '#F58518'])
plt.title('Distribucion por split - Task A')
plt.xlabel('Split')
plt.ylabel('Cantidad')
plt.tight_layout()
plt.show()

ResultadoTaskA = entrenar_evaluar_decision_tree(
    DatosTaskA,
    columna_target='is_clone',
    etiquetas=[0, 1],
    seed=SEED,
    estrategia_balanceo=ESTRATEGIA_BALANCEO,
    graficar_roc=True,
)


## 7) Task B: `clone_type` (`type_III` vs `type_IV`)

Aqu? usamos solo pares positivos (`is_clone == 1`), se balancea train y se entrena Decision Tree.


In [ ]:
DatosPositivos = DatosPreparados[DatosPreparados['is_clone'] == 1].copy()

SplitTaskB = grouped_train_val_test_split(
    df=DatosPositivos,
    group_col='problem_id',
    target_col='clone_type',
    seed=SEED + 100,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
)

DatosTaskB = asignar_split(
    DatosPositivos,
    idx_train=SplitTaskB.train_idx,
    idx_val=SplitTaskB.val_idx,
    idx_test=SplitTaskB.test_idx,
)

print('Estadisticas split Task B:')
StatsB = split_statistics(DatosTaskB, 'split', 'clone_type', 'problem_id')
for fila in StatsB:
    print(fila)

conteos_split_b = DatosTaskB.groupby(['split', 'clone_type']).size().unstack(fill_value=0)
conteos_split_b.plot(kind='bar', figsize=(8,4), color=['#72B7B2', '#E45756'])
plt.title('Distribucion por split - Task B')
plt.xlabel('Split')
plt.ylabel('Cantidad')
plt.tight_layout()
plt.show()

EtiquetasTaskB = sorted(DatosTaskB['clone_type'].unique().tolist())

ResultadoTaskB = entrenar_evaluar_decision_tree(
    DatosTaskB,
    columna_target='clone_type',
    etiquetas=EtiquetasTaskB,
    seed=SEED,
    estrategia_balanceo=ESTRATEGIA_BALANCEO,
    graficar_roc=False,
)


## 8) Resumen final de resultados

Se muestran las m?tricas principales de validaci?n y test para ambas tareas.


In [ ]:
ResumenFinal = pd.DataFrame([
    {
        'tarea': 'Task A - is_clone',
        'accuracy_val': ResultadoTaskA['metricas_val']['accuracy'],
        'f1_macro_val': ResultadoTaskA['metricas_val']['f1_macro'],
        'accuracy_test': ResultadoTaskA['metricas_test']['accuracy'],
        'f1_macro_test': ResultadoTaskA['metricas_test']['f1_macro'],
    },
    {
        'tarea': 'Task B - clone_type',
        'accuracy_val': ResultadoTaskB['metricas_val']['accuracy'],
        'f1_macro_val': ResultadoTaskB['metricas_val']['f1_macro'],
        'accuracy_test': ResultadoTaskB['metricas_test']['accuracy'],
        'f1_macro_test': ResultadoTaskB['metricas_test']['f1_macro'],
    },
])
ResumenFinal


## 9) Conclusi?n r?pida

- El baseline queda ejecutado dentro del notebook, con visualizaci?n directa de m?tricas y matrices.
- Para an?lisis m?s profundo o comparaci?n hist?rica, se puede exportar artefactos; para revisi?n tipo Colab, este notebook es suficiente.
